# DiCo-NLI Track 1 — Assignment 2 experiments (Colab)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Run the cells in order. Everything (code, data, official scorer) comes from the GitHub clone in cell 1.

Models are trained in **fp32** (DeBERTa-v3 gives NaN in fp16 and on Apple MPS).

In [ ]:
# 1. Setup: clone the repo (data, official scorer, src/data.py)
!git clone -q https://github.com/MuneebBaig71/DiCo-NLI_SemEval.git
%cd DiCo-NLI_SemEval
!pip -q install sentencepiece
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. Check the A2 files came with the clone
from pathlib import Path
from google.colab import files
need = ["src/decoding.py", "src/train_a2.py", "06_decoding_experiments.py"]
missing = [p for p in need if not Path(p).exists()]
print("MISSING:", missing if missing else "none - all A2 files present")

# software versions (reported in Table 4 of the A2 report)
import sys, torch, transformers, sklearn
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__, "| sklearn", sklearn.__version__)


In [ ]:
# 3. (Recommended) save results to Google Drive so a disconnect doesn't lose them
from google.colab import drive
drive.mount("/content/drive")
DRIVE_OUT = "/content/drive/MyDrive/DiCo_A2_results"

In [ ]:
# 4. Smoke test (~2 min): 1 epoch of DeBERTa-v3 on 300 rows, with the consistency loss on.
#    Checks: no NaN, the loss terms print, files get written. Not a real result.
import importlib, src.decoding, src.train_a2
importlib.reload(src.decoding); importlib.reload(src.train_a2)
from src.train_a2 import run
meta = run(model="microsoft/deberta-v3-base", epochs=1, max_train_rows=300,
           lambda_cons=1.0, tag="smoke", out_dir="results_smoke")
print(meta["history"])

In [ ]:
# 5. Main runs: every run reported in the A2 report (13 runs; ~45 GPU-minutes on a T4).
#    M0 = A1 RoBERTa rerun under the A2 protocol (fair comparison)
#    M1 = plain DeBERTa-v3, M2 = + consistency loss, M3 = M2 + Neg weighting  (seeds 13, 42, 87)
#    M2b = consistency weight 0.5, seed 13 only (sensitivity check)
#    Finished runs are skipped, so this cell can be re-run after a disconnect.
import json, shutil, pandas as pd
from pathlib import Path
import importlib, src.train_a2; importlib.reload(src.train_a2)
from src.train_a2 import run

EXPS = [
    (dict(tag="M0_roberta",          model="roberta-base"),                                     [13, 42, 87]),
    (dict(tag="M1_deberta",          model="microsoft/deberta-v3-base"),                        [13, 42, 87]),
    (dict(tag="M2_deberta_cons",     model="microsoft/deberta-v3-base", lambda_cons=1.0),       [13, 42, 87]),
    (dict(tag="M3_deberta_cons_neg", model="microsoft/deberta-v3-base", lambda_cons=1.0, neg_weight=2.0), [13, 42, 87]),
    (dict(tag="M2b_deberta_cons05",  model="microsoft/deberta-v3-base", lambda_cons=0.5),       [13]),
]

for e, seeds in EXPS:
    for seed in seeds:
        tag = f"{e['tag']}_seed{seed}"
        if Path(f"results_a2/runs/{tag}.json").exists():
            print("skip (done):", tag); continue
        run(**{**e, "tag": tag, "seed": seed})
        try:   # copy after every run, so a disconnect loses at most one run
            shutil.copytree("results_a2", DRIVE_OUT, dirs_exist_ok=True)
        except NameError:
            pass


In [ ]:
# 6. Score every run on dev with the official scorer, under all 4 decoding rules
!python 06_decoding_experiments.py

In [ ]:
# 7. Training summary (best epoch chosen on the validation split, not dev)
rows = []
for f in sorted(Path("results_a2/runs").glob("*.json")):
    m = json.loads(f.read_text())
    rows.append(dict(tag=m["tag"], best_epoch=m["best_epoch"],
                     best_val_wf1=round(m["best_val_wf1"], 4), minutes=round(m["seconds"] / 60, 1)))
pd.DataFrame(rows)

In [ ]:
# 8. Download everything as a zip (unzip into your Mac folder)
!zip -qr results_a2.zip results_a2
files.download("results_a2.zip")

In [ ]:
# 9. Build the Codabench submission (Track 1 / English): M3 three-seed ensemble + struct_full decoding.
#    This is Codabench submission 1042178 (dev W-F1 0.936, SoftCons 1.0, HardCons 0.924).
import shutil, zipfile, subprocess
import numpy as np, pandas as pd
from pathlib import Path
from src.data import split_path, score, headline, LABELS
from src.decoding import partner_index, decode

# bring back runs trained in earlier sessions (no-op if already present)
if Path("/content/drive/MyDrive/DiCo_A2_results").exists():
    shutil.copytree("/content/drive/MyDrive/DiCo_A2_results", "results_a2", dirs_exist_ok=True)

SYSTEM, SEEDS, MODE = "M3_deberta_cons_neg", [13, 42, 87], "struct_full"

# 1) average the seeds' probabilities, aligned to the submission template order
template = pd.read_csv(split_path("dev", "submission_template"), keep_default_na=False)
ids = template.instance_id.tolist()
frames = [pd.read_csv(f"results_a2/predictions/{SYSTEM}_seed{s}_dev_probs.csv").set_index("instance_id")
          for s in SEEDS]
probs = np.mean([f.loc[ids, LABELS].to_numpy() for f in frames], axis=0)

# 2) decode (partners come from pair_id in the participant file, never from gold labels)
participant = pd.read_csv(split_path("dev", "participant_labeled"), keep_default_na=False)
preds = decode(probs, partner_index(ids, participant), MODE)
Path("submission").mkdir(exist_ok=True)
csv = Path("submission/track1_predictions.csv")
pd.DataFrame({"instance_id": ids, "label": preds}).to_csv(csv, index=False)

# 3) check the Codabench format rules
df = pd.read_csv(csv, keep_default_na=False)
assert list(df.columns) == ["instance_id", "label"]
assert df.instance_id.is_unique and set(df.instance_id) == set(template.instance_id)
assert set(df.label) <= set(LABELS) and len(df) == len(template)
print("format OK:", len(df), "rows")

# 4) local score with the official scorer (matches Codabench on the dev phase)
print(headline(score(csv, "dev")))

# 5) zip with the file at the archive root
zip_path = Path("submission.zip"); zip_path.unlink(missing_ok=True)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(csv, arcname="track1_predictions.csv")
print(subprocess.run(["unzip", "-Z1", str(zip_path)], capture_output=True, text=True).stdout)
from google.colab import files
files.download(str(zip_path))


In [ ]:
# 10. Checkpoint save/load check (~5 min, A2 brief section 8).
#     Fresh runtime is fine: run cells 1 and 3 first, then this cell.
#     Trains M3 seed 42 once with save_checkpoint=True, reloads the saved model from disk,
#     predicts dev again, and checks the reloaded predictions match the trained model's.
import importlib, json, shutil, numpy as np, pandas as pd, torch, transformers
from pathlib import Path
import src.train_a2; importlib.reload(src.train_a2)
from src.train_a2 import run, predict_from_checkpoint
from src.data import LABELS, split_path, score, headline

TAG, OUT = "M3_ckpt_check_seed42", "results_ckpt"
meta = run(model="microsoft/deberta-v3-base", lambda_cons=1.0, neg_weight=2.0, seed=42,
           tag=TAG, out_dir=OUT, save_checkpoint=True)
ckpt = f"{OUT}/checkpoints/{TAG}"

saved = pd.read_csv(f"{OUT}/predictions/{TAG}_dev_probs.csv").set_index("instance_id")
ids, reloaded = predict_from_checkpoint(ckpt, "dev")
diff = np.abs(saved.loc[ids, LABELS].to_numpy() - reloaded).max()
same_labels = (saved.loc[ids, LABELS].to_numpy().argmax(1) == reloaded.argmax(1)).mean()

pd.DataFrame({"instance_id": ids, "label": [LABELS[k] for k in reloaded.argmax(1)]}
             ).to_csv(f"{OUT}/reloaded_dev.csv", index=False)
print("best epoch:", meta["best_epoch"], "| train minutes:", round(meta["seconds"] / 60, 1))
print("checkpoint files:", sorted(p.name for p in Path(ckpt).iterdir()))
print("checkpoint size (MB):", round(sum(p.stat().st_size for p in Path(ckpt).iterdir()) / 1e6))
print("max |prob diff| trained vs reloaded:", diff)
print("identical argmax labels:", same_labels)
print("reloaded model, argmax, official scorer:", headline(score(f"{OUT}/reloaded_dev.csv", "dev")))
print("versions: torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPU", torch.cuda.get_device_name(0))

# compare with the original M3 seed-42 run (same seed and config, separate training run)
orig = pd.read_csv(f"/content/drive/MyDrive/DiCo_A2_results/predictions/M3_deberta_cons_neg_seed42_dev_probs.csv").set_index("instance_id")
print("max |prob diff| vs original seed-42 run:", np.abs(orig.loc[ids, LABELS].to_numpy() - reloaded).max(),
      "| label agreement:", (orig.loc[ids, LABELS].to_numpy().argmax(1) == reloaded.argmax(1)).mean())

shutil.copytree(OUT, "/content/drive/MyDrive/DiCo_A2_ckpt_check", dirs_exist_ok=True)
print("copied to Drive: MyDrive/DiCo_A2_ckpt_check")